# Домашняя работа 3. Анализ датасета Titanic

## 1. Чтение датасета

In [7]:
"""Задача 1. Чтение датасета Titanic.

Файл train.csv лежит или рядом с ноутбуком (так бывает в Colab), или в общей
папке DataSet/ (так он лежит в репозитории), поэтому путь ищется, а не
вписывается жёстко.
"""

import re
from pathlib import Path

import pandas as pd


def find_dataset(filename: str = "train.csv") -> Path:
    """Ищет файл датасета в текущей папке и вверх по дереву каталогов."""
    for folder in [Path.cwd(), *Path.cwd().parents]:
        for candidate in (folder / filename, folder / "DataSet" / filename):
            if candidate.exists():
                return candidate
    raise FileNotFoundError(f"не найден файл {filename}")


# PassengerId — идентификатор пассажира, а не признак, поэтому делаем его индексом
df = pd.read_csv(find_dataset(), index_col="PassengerId")

print(f"строк: {df.shape[0]}, столбцов: {df.shape[1]}")
df.head()

строк: 891, столбцов: 11


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Основная информация о датасете

In [8]:
# df.info() печатает типы данных, число непустых значений и объём памяти
df.info()

# Пропуски: сколько NaN в каждом столбце и какой это процент от всех строк
counts = df.isna().sum()
missing = pd.DataFrame({"пропусков": counts, "процент": (counts / len(df) * 100).round(2)})
missing[missing["пропусков"] > 0]

<class 'pandas.core.frame.DataFrame'>
Index: 891 entries, 1 to 891
Data columns (total 11 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Name      891 non-null    object 
 3   Sex       891 non-null    object 
 4   Age       714 non-null    float64
 5   SibSp     891 non-null    int64  
 6   Parch     891 non-null    int64  
 7   Ticket    891 non-null    object 
 8   Fare      891 non-null    float64
 9   Cabin     204 non-null    object 
 10  Embarked  889 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 83.5+ KB


,пропусков,процент
Age,177,19.87
Cabin,687,77.10
Embarked,2,0.22


In [9]:
# describe() по умолчанию считает статистику только по числовым столбцам:
# количество, среднее, стандартное отклонение, минимум, квартили и максимум
df.describe().round(2)

,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.00,891.00,714.00,891.00,891.00,891.00
mean,0.38,2.31,29.70,0.52,0.38,32.20
std,0.49,0.84,14.53,1.10,0.81,49.69
min,0.00,1.00,0.42,0.00,0.00,0.00
25%,0.00,2.00,20.12,0.00,0.00,7.91
50%,0.00,3.00,28.00,0.00,0.00,14.45
75%,1.00,3.00,38.00,1.00,0.00,31.00
max,1.00,3.00,80.00,8.00,6.00,512.33


## 3. Процент выживаемости по классам

In [10]:
# Survived хранит 0 и 1, поэтому среднее по группе — это доля выживших.
# Умножаем на 100, чтобы получить проценты, и округляем до сотых.
survival = (df.groupby("Pclass")["Survived"].mean() * 100).round(2)
survival.rename("выживших, %").to_frame()

,"выживших, %"
Pclass,
1,62.96
2,47.28
3,24.24


## 4. Самое популярное мужское и женское имя

In [11]:
def first_name(full_name: str) -> str:
    """Достаёт личное имя пассажира из строки вида «Фамилия, Титул. Имя Отчество».

    Полные имена в датасете уникальны (891 значение на 891 строку), поэтому
    «популярное имя» приходится извлекать из строки. Порядок действий:
      1. отрезаем всё до первой точки — там остаются фамилия и титул;
      2. у замужних женщин записано имя мужа, а собственное имя стоит в скобках
         («Cumings, Mrs. John Bradley (Florence Briggs Thayer)» → Florence),
         поэтому содержимое скобок имеет приоритет;
      3. скобки с прозвищем в кавычках («Mr. Pastcho ("Pentcho")») пропускаем;
      4. из того, что осталось, берём первое слово — это и есть имя.
    """
    # 1. всё после титула
    tail = full_name.split(".", 1)[1] if "." in full_name else full_name

    # 2-3. содержимое скобок, если это не прозвище в кавычках
    in_parentheses = re.search(r"\(([^)]*)\)", tail)
    if in_parentheses and not in_parentheses.group(1).strip().startswith('"'):
        tail = in_parentheses.group(1)

    # убираем прозвища в кавычках и оставшиеся скобки
    tail = re.sub(r'"[^"]*"', " ", tail)
    tail = re.sub(r"\([^)]*\)", " ", tail)

    # 4. первое слово
    words = tail.split()
    return words[0] if words else ""


# apply() прогоняет функцию по каждой строке столбца Name
df["FirstName"] = df["Name"].apply(first_name)

# value_counts() сортирует по частоте: первая строка — самое популярное имя
for sex, label in (("male", "мужское"), ("female", "женское")):
    counts = df.loc[df["Sex"] == sex, "FirstName"].value_counts()
    print(f"{label}: {counts.index[0]} — {counts.iloc[0]} чел.")

мужское: William — 35 чел.
женское: Anna — 15 чел.


## 5. Самое популярное имя каждого пола в каждом классе

In [12]:
# Тот же приём, но срез делаем не по всему датасету, а по каждому классу:
# groupby("Pclass") отдаёт три части, в каждой отдельно считаем популярные
# имена мужчин и женщин.
rows = []
for pclass, part in df.groupby("Pclass"):
    for sex, label in (("male", "мужское"), ("female", "женское")):
        counts = part.loc[part["Sex"] == sex, "FirstName"].value_counts()
        rows.append({
            "класс": pclass,
            "пол": label,
            "имя": counts.index[0],
            "человек": counts.iloc[0],
        })

pd.DataFrame(rows).set_index(["класс", "пол"])

имя  человек
класс пол                        
1     мужское    William       11
      женское  Elizabeth        5
2     мужское    William        9
      женское  Elizabeth        5
3     мужское    William       15
      женское       Anna        9

## 6. Пассажиры старше 44 лет

In [13]:
# Булева маска: df["Age"] > 44 — это столбец из True/False, а df[маска]
# оставляет только строки, где True. Пассажиры с неизвестным возрастом
# в выборку не попадут: сравнение NaN с числом всегда даёт False.
older = df[df["Age"] > 44]
print(f"пассажиров старше 44 лет: {len(older)}")
older.head(10)

пассажиров старше 44 лет: 115


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
PassengerId,,,,,,,,,,,,
7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,Timothy
12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S,Elizabeth
16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.0,0,0,248706,16.0000,NaN,S,Mary
34,0,2,"Wheadon, Mr. Edward H",male,66.0,0,0,C.A. 24579,10.5000,NaN,S,Edward
53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C,Myna
55,0,1,"Ostby, Mr. Engelhart Cornelius",male,65.0,0,1,113509,61.9792,B30,C,Engelhart
63,0,1,"Harris, Mr. Henry Birkhardt",male,45.0,1,0,36973,83.4750,C83,S,Henry
93,0,1,"Chaffee, Mr. Herbert Fuller",male,46.0,1,0,W.E.P. 5734,61.1750,E31,S,Herbert
95,0,3,"Coxon, Mr. Daniel",male,59.0,0,0,364500,7.2500,NaN,S,Daniel


## 7. Мужчины младше 44 лет

In [14]:
# Два условия объединяем через & (поэлементное «и»): нужны одновременно
# мужчины и возраст меньше 44. Каждое условие — в своих скобках, потому что
# у & приоритет выше, чем у операторов сравнения.
younger_males = df[(df["Sex"] == "male") & (df["Age"] < 44)]
print(f"мужчин младше 44 лет: {len(younger_males)}")
younger_males.head(10)

мужчин младше 44 лет: 368


,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
PassengerId,,,,,,,,,,,,
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.250,NaN,S,Owen
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.050,NaN,S,William
8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.075,NaN,S,Gosta
13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.050,NaN,S,William
14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.275,NaN,S,Anders
17,0,3,"Rice, Master. Eugene",male,2.0,4,1,382652,29.125,NaN,Q,Eugene
21,0,2,"Fynney, Mr. Joseph J",male,35.0,0,0,239865,26.000,NaN,S,Joseph
22,1,2,"Beesley, Mr. Lawrence",male,34.0,0,0,248698,13.000,D56,S,Lawrence
24,1,1,"Sloper, Mr. William Thompson",male,28.0,0,0,113788,35.500,A6,S,William


## 8. Количество n-местных кают

In [15]:
# Первый value_counts() считает, сколько человек приходится на каждую запись
# о каюте, второй — сколько кают имеют одинаковое число жильцов.
# dropna() убирает 687 пассажиров, у которых каюта не указана.
per_cabin = df["Cabin"].dropna().value_counts()
sizes = per_cabin.value_counts().sort_index()

# У 24 пассажиров записано несколько кают сразу («C23 C25 C27») — это семья,
# выкупившая смежные каюты одной бронью: считаем такую запись единым размещением.
sizes = sizes[sizes.index >= 2]
sizes.index.name = "человек в каюте"
sizes.rename("количество кают").to_frame()

,количество кают
человек в каюте,
2,38
3,5
4,3
